In [ ]:
# 日本語対応中型LLMのトレーニング・パッケージ化ノートブック
import os
import json
import torch
import numpy as np
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments, pipeline
import subprocess
import sys
import zipfile
from tqdm import tqdm

# ============================================================================
# 1. インストール
# ============================================================================
print("[1] パッケージのインストール...")
!pip install -q transformers datasets torch sentencepiece fugashi mecab-python3 unidic-lite
print("✓ インストール完了")

# ============================================================================
# 2. 日本語データセットの準備
# ============================================================================
print("\n[2] 日本語トレーニングデータの準備...")

# 日本語データセットの生成（簡易版）
japanese_texts = [
    "これは日本語のサンプルテキストです。自然言語処理について学習しています。",
    "機械学習とディープラーニングは現代のAIの基礎です。",
    "Transformerモデルは自然言語処理で優れた性能を発揮します���",
    "日本語の形態素解析は固有表現抽出に重要です。",
    "大規模言語モデルは様々なタスクに応用されています。",
    "データの品質がモデルの性能を大きく左右します。",
    "ファインチューニングは事前学習済みモデルを特定のタスクに適応させます。",
    "GPUを使用することで学習を高速化できます。",
    "日本語テキストの前処理には複数の手法があります。",
    "言語モデルの評価指標にはPPL（Perplexity）が使用されます。",
]

# テキストファイルの作成
data_dir = Path("/tmp/japanese_llm_data")
data_dir.mkdir(exist_ok=True)
train_file = data_dir / "train.txt"
with open(train_file, "w", encoding="utf-8") as f:
    for _ in range(50):  # データ数を増やす
        for text in japanese_texts:
            f.write(text + "\n")

print(f"✓ トレーニングデータ作成: {train_file}")
print(f"  ファイルサイズ: {train_file.stat().st_size / 1024:.1f} KB")

# ============================================================================
# 3. 中型モデルの選択とトークナイザーの準備
# ============================================================================
print("\n[3] 中型モデルの準備...")

# 日本語対応の中型言語モデル
model_name = "rinna/japanese-gpt2-medium"  # 330M パラメータの日本語GPT2
print(f"モデル: {model_name}")
print(f"パラメータ数: 約330M (中型)")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"デバイス: {device}")

# トークナイザーとモデルの読み込み
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32)
model.to(device)
print(f"✓ モデル読み込み完了 ({model.num_parameters() / 1e6:.0f}M parameters)")

# ============================================================================
# 4. データセットの準備
# ============================================================================
print("\n[4] データセットの準備...")

train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path=str(train_file),
    block_size=128
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(f"✓ データセット準備完了")
print(f"  サンプル数: {len(train_dataset)}")
print(f"  ブロックサイズ: 128")

# ============================================================================
# 5. トレーニング設定
# ============================================================================
print("\n[5] トレーニング設定...")

output_dir = Path("/tmp/japanese_llm_model")
output_dir.mkdir(exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    save_steps=50,
    save_total_limit=2,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    fp16=device == "cuda",
    report_to="none",
    seed=42
)

print("✓ トレーニング設定完了")
print(f"  エポック: {training_args.num_train_epochs}")
print(f"  バッチサイズ: {training_args.per_device_train_batch_size}")
print(f"  学習率: {training_args.learning_rate}")

# ============================================================================
# 6. トレーニング実行
# ============================================================================
print("\n[6] トレーニング開始...")

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

train_result = trainer.train()
print(f"✓ トレーニング完了")
print(f"  最終損失: {train_result.training_loss:.4f}")

# ============================================================================
# 7. モデルの保存
# ============================================================================
print("\n[7] モデルの保存...")

final_model_dir = Path("/tmp/japanese_llm_final")
final_model_dir.mkdir(exist_ok=True)

model.save_pretrained(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

# config.json に学習情報を追加
config_path = final_model_dir / "config.json"
with open(config_path, "r") as f:
    config = json.load(f)
config["training_info"] = {
    "model_name": model_name,
    "trained_on": "Japanese texts",
    "training_loss": float(train_result.training_loss),
    "epochs": training_args.num_train_epochs
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"✓ モデル保存完了: {final_model_dir}")

# ============================================================================
# 8. パッケージ化（ZIPファイル作成）
# ============================================================================
print("\n[8] パッケージ化...")

package_path = Path("/tmp/japanese_llm_package.zip")
with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in final_model_dir.rglob("*"):
        if file.is_file():
            arcname = file.relative_to(final_model_dir.parent)
            zipf.write(file, arcname)

print(f"✓ パッケージ化完了")
print(f"  ZIPファイル: {package_path}")
print(f"  ファイルサイズ: {package_path.stat().st_size / 1024 / 1024:.1f} MB")

# ============================================================================
# 9. パッケージ情報ファイルの作成
# ============================================================================
print("\n[9] パッケージ情報の作成...")

package_info = {
    "name": "japanese-llm-trained",
    "version": "1.0.0",
    "description": "Japanese LLM (GPT2-medium) trained on Japanese texts",
    "base_model": model_name,
    "parameters": "330M",
    "language": "Japanese",
    "training": {
        "epochs": int(training_args.num_train_epochs),
        "batch_size": int(training_args.per_device_train_batch_size),
        "learning_rate": float(training_args.learning_rate),
        "final_loss": float(train_result.training_loss)
    },
    "usage": {
        "load": "from transformers import AutoTokenizer, AutoModelForCausalLM\ntokenizer = AutoTokenizer.from_pretrained('path/to/model')\nmodel = AutoModelForCausalLM.from_pretrained('path/to/model')",
        "inference": "pipe = pipeline('text-generation', model=model, tokenizer=tokenizer)\nresult = pipe('こんにちは', max_length=50)"
    }
}

info_path = Path("/tmp/japanese_llm_final/package_info.json")
with open(info_path, "w", encoding="utf-8") as f:
    json.dump(package_info, f, indent=2, ensure_ascii=False)

print(f"✓ パッケージ情報作成完了")

# ============================================================================
# 10. 推論テスト
# ============================================================================
print("\n[10] 推論テスト...")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)

test_prompts = [
    "日本語の",
    "自然言語処理は",
    "機械学習の"
]

print("生成テスト:")
for prompt in test_prompts:
    output = generator(prompt, max_length=30, num_return_sequences=1, do_sample=True)
    print(f"  入力: {prompt}")
    print(f"  出力: {output[0]['generated_text']}")

# ============================================================================
# 11. 最終統計
# ============================================================================
print("\n" + "="*70)
print("トレーニング・パッケージ化完了")
print("="*70)
print(f"\nモデル統計:")
print(f"  ベースモデル: {model_name}")
print(f"  パラメータ数: {model.num_parameters() / 1e6:.0f}M")
print(f"  最終損失: {train_result.training_loss:.4f}")
print(f"\n保存先:")
print(f"  モデルディレクトリ: {final_model_dir}")
print(f"  パッケージ: {package_path}")
print(f"  パッケージサイズ: {package_path.stat().st_size / 1024 / 1024:.1f} MB")
print(f"\nKaggleへのアップロード:")
print(f"  1. Kaggle Datasetsにアップロード")
print(f"  2. kaggle datasets create -p {package_path.parent}")
print(f"\nモデルの使用:")
print(f"  from transformers import AutoTokenizer, AutoModelForCausalLM")
print(f"  tokenizer = AutoTokenizer.from_pretrained('path/to/model')")
print(f"  model = AutoModelForCausalLM.from_pretrained('path/to/model')")
print("="*70)